# LeetCode 1296 — Divide Array in Sets of K Consecutive Numbers bằng Python

Link đề: https://leetcode.com/problems/divide-array-in-sets-of-k-consecutive-numbers/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def isPossibleDivide(self, nums: list[int], k: int) -> bool:
        ...
```

> Bài này giống hệt LC 846 *Hand of Straights*.

## 1. Mục tiêu học

1. Hiểu **lập luận tham lam then chốt**: số **nhỏ nhất** còn lại **bắt buộc** phải là phần tử đầu của một nhóm `x, x+1, …, x+k−1`.
2. Dùng **`Counter`** để đếm và "rút" cả loạt phần tử cùng lúc (rút `cnt[x]` nhóm một lần thay vì từng nhóm).
3. Đi từ brute force `O(n²)` lên `O(n log n)` (sort khoá / heap), rồi `O(n)` kỳ vọng bằng cách **chỉ bắt đầu từ các khoá đầu chuỗi** (không cần sort).
4. Biết thêm kỹ thuật một lượt "**số nhóm đang mở**" với hàng đợi.
5. Tránh lỗi: duyệt khoá không theo thứ tự tăng, chỉ trừ 1 thay vì `cnt[x]`, `KeyError` với `dict`, kiểm tra điều kiện cần mà tưởng là đủ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên dương `k`. Hỏi có thể **chia toàn bộ mảng** thành các nhóm, mỗi nhóm gồm **`k` số nguyên liên tiếp** hay không? Trả `True` nếu được, ngược lại `False`.

**Ví dụ 1:** `nums = [1,2,3,3,4,4,5,6], k = 4` → `True` (`[1,2,3,4]` và `[3,4,5,6]`).

**Ví dụ 2:** `nums = [3,2,1,2,3,4,3,4,5,9,10,11], k = 3` → `True` (`[1,2,3]`, `[2,3,4]`, `[3,4,5]`, `[9,10,11]`).

**Ví dụ 3:** `nums = [1,2,3,4], k = 3` → `False` (4 phần tử không chia hết thành nhóm 3).

**Ràng buộc:** `1 <= k <= nums.length <= 10^5`, `1 <= nums[i] <= 10^9`.

In [1]:
import heapq
import random
import time
from collections import Counter, deque
from itertools import combinations
from typing import List

examples = [
    ([1, 2, 3, 3, 4, 4, 5, 6], 4, True),
    ([3, 2, 1, 2, 3, 4, 3, 4, 5, 9, 10, 11], 3, True),
    ([1, 2, 3, 4], 3, False),
]
for nums, k, want in examples:
    print(f"nums={nums}, k={k} -> mong đợi {want}")

nums=[1, 2, 3, 3, 4, 4, 5, 6], k=4 -> mong đợi True
nums=[3, 2, 1, 2, 3, 4, 3, 4, 5, 9, 10, 11], k=3 -> mong đợi True
nums=[1, 2, 3, 4], k=3 -> mong đợi False


## 3. Kiến thức nền

### 3.1 `Counter` — đếm tần suất

`Counter(nums)` là dict `giá trị → số lần`. Truy cập khoá không có trả `0` (không lỗi, **không** thêm khoá). Có thể trừ trực tiếp `cnt[x] -= 3`.

In [2]:
cnt = Counter([1, 2, 3, 3, 4, 4, 5, 6])
print(cnt)
print("cnt[3] =", cnt[3], "| cnt[100] =", cnt[100], "| 100 có trong cnt?", 100 in cnt)
cnt[3] -= 1
print("sau khi trừ:", cnt[3])

Counter({3: 2, 4: 2, 1: 1, 2: 1, 5: 1, 6: 1})
cnt[3] = 2 | cnt[100] = 0 | 100 có trong cnt? False
sau khi trừ: 1


### 3.2 Sắp xếp các khoá

`sorted(cnt)` trả các giá trị **khác nhau** theo thứ tự tăng. Duyệt khoá của dict trực tiếp thì theo **thứ tự chèn** (thứ tự xuất hiện lần đầu trong mảng), **không** phải tăng dần.

In [3]:
nums = [3, 2, 1, 2, 3, 4]
cnt = Counter(nums)
print("thứ tự chèn:", list(cnt), "| sorted:", sorted(cnt))

thứ tự chèn: [3, 2, 1, 4] | sorted: [1, 2, 3, 4]


### 3.3 Min-heap với `heapq`

`heapq.heapify(a)` biến list thành heap; `a[0]` luôn là phần tử nhỏ nhất; `heappop` lấy nhỏ nhất trong `O(log n)`.

In [4]:
h = [5, 1, 4, 2]
heapq.heapify(h)
print("nhỏ nhất:", h[0])
print([heapq.heappop(h) for _ in range(len(h))])

nhỏ nhất: 1
[1, 2, 4, 5]


### 3.4 `set` để kiểm tra "đầu chuỗi"

`x` là **đầu một chuỗi số liên tiếp** khi `x − 1` không xuất hiện. Kiểm tra trong `O(1)` với `set` hoặc với `Counter` (toán tử `in`).

In [5]:
s = set([1, 2, 3, 9, 10, 20])
print("các đầu chuỗi:", sorted(x for x in s if x - 1 not in s))

các đầu chuỗi: [1, 9, 20]


### 3.5 `deque` làm hàng đợi trượt

Dùng `deque` với `append` và `popleft` để giữ "số nhóm mở tại mỗi vị trí trong `k` bước gần nhất" (Cách 5).

In [6]:
d = deque()
for x in [2, 0, 1, 3]:
    d.append(x)
    if len(d) > 2:
        print("rời cửa sổ:", d.popleft())
print("còn lại:", list(d))

rời cửa sổ: 2
rời cửa sổ: 0
còn lại: [1, 3]


## 4. Ý tưởng nền tảng / trực giác

Làm tay **Ví dụ 1** `[1,2,3,3,4,4,5,6]`, `k = 4`. Đếm: `{1:1, 2:1, 3:2, 4:2, 5:1, 6:1}`.

- Số nhỏ nhất là `1`. Số `1` không thể đứng giữa hay cuối một nhóm (không có `0`), nên **bắt buộc** mở nhóm `[1,2,3,4]`. Đếm còn: `{3:1, 4:1, 5:1, 6:1}`.
- Số nhỏ nhất còn lại là `3` ⇒ bắt buộc nhóm `[3,4,5,6]`. Hết ⇒ `True`.

Nếu số nhỏ nhất `x` xuất hiện `c` lần, thì **cả `c` bản sao** đều phải là đầu nhóm ⇒ mở `c` nhóm cùng lúc: cần `cnt[x+1], …, cnt[x+k−1]` đều `≥ c`.

**Ví dụ 3** `[1,2,3,4]`, `k = 3`: mở `[1,2,3]`, còn `[4]` ⇒ số nhỏ nhất `4` cần `5, 6` ⇒ không có ⇒ `False`. (Hoặc đơn giản: `4 % 3 ≠ 0`.)

In [7]:
def hand_trace(nums, k):
    cnt = Counter(nums)
    for x in sorted(cnt):
        c = cnt[x]
        if c == 0:
            continue
        group = list(range(x, x + k))
        ok = all(cnt[t] >= c for t in group)
        print(f"nhỏ nhất còn lại = {x} (x{c}) -> cần {c} nhóm {group}: {'được' if ok else 'KHÔNG được'}")
        if not ok:
            return False
        for t in group:
            cnt[t] -= c
    return True

print(hand_trace(*examples[0][:2]))

nhỏ nhất còn lại = 1 (x1) -> cần 1 nhóm [1, 2, 3, 4]: được
nhỏ nhất còn lại = 3 (x1) -> cần 1 nhóm [3, 4, 5, 6]: được
True


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** sắp xếp mảng thành list; lặp: lấy phần tử nhỏ nhất `x` (đầu list), **xoá** lần lượt `x, x+1, …, x+k−1` khỏi list (`list.remove`); nếu thiếu số nào ⇒ `False`. List rỗng ⇒ `True`.

In [8]:
def divide_brute(nums, k):
    if len(nums) % k:
        return False
    arr = sorted(nums)
    while arr:
        x = arr[0]
        for t in range(x, x + k):
            if t in arr:               # O(n)
                arr.remove(t)          # O(n)
            else:
                return False
    return True

for nums, k, want in examples:
    print(divide_brute(nums, k), "| mong đợi", want)

True | mong đợi True
True | mong đợi True
False | mong đợi False


**Độ phức tạp:** có `n/k` nhóm, mỗi nhóm `k` lần `remove` tốn `O(n)` ⇒ `O(n²)`. Với `n = 10^5` là `10^10` — quá chậm.

**Nút thắt:** xoá từng phần tử khỏi list và tìm kiếm tuyến tính. Dùng `Counter` thì "xoá" chỉ là `cnt[t] -= 1` trong `O(1)`, và có thể trừ **cả loạt** `cnt[x]` nhóm một lần.

In [9]:
rng0 = random.Random(0)
for n in [2000, 4000, 8000]:
    kk = n // 4
    nums = list(range(1, kk + 1)) * 4     # chia được thành 4 nhóm, mỗi nhóm k = n/4 số
    rng0.shuffle(nums)
    t = time.perf_counter()
    divide_brute(nums, kk)
    print(f"n={n}, k={kk}: brute force {time.perf_counter() - t:.3f}s")

n=2000, k=500: brute force 0.010s
n=4000, k=1000: brute force 0.037s


n=8000, k=2000: brute force 0.168s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tham lam):** gọi `x` là giá trị nhỏ nhất còn lại, xuất hiện `c = cnt[x]` lần. Mỗi bản sao của `x` phải thuộc một nhóm, và trong nhóm đó `x` là **phần tử nhỏ nhất** (vì không còn số nào nhỏ hơn) ⇒ nhóm là `[x, x+1, …, x+k−1]`. Vậy **bắt buộc** mở đúng `c` nhóm như thế.

**Quy tắc "nếu… thì…":**
- Nếu `len(nums) % k != 0` ⇒ `False` ngay.
- Nếu `cnt[x+i] < c` với một `i ∈ [1, k−1]` ⇒ `False`.
- Ngược lại trừ `c` khỏi `cnt[x], …, cnt[x+k−1]` rồi tìm số nhỏ nhất kế tiếp.

**Invariant:** trước mỗi bước, multiset còn lại chia được ⇔ multiset ban đầu chia được (vì các nhóm ta vừa lấy là **bắt buộc** trong mọi cách chia).

**Ba cách tìm "số nhỏ nhất còn lại":** sort khoá một lần (Cách 2), heap (Cách 3), hoặc nhận ra rằng chỉ cần đi từ **đầu mỗi chuỗi liên tiếp** (Cách 4, không sort).

## 7. Các cách tiếp theo

### Cách 2 — `Counter` + duyệt khoá đã sắp xếp

**Từng đoạn:**
1. Kiểm tra `n % k`.
2. `cnt = Counter(nums)`.
3. Với `x` trong `sorted(cnt)`: nếu `cnt[x] > 0` thì `c = cnt[x]`; với `t = x … x+k−1`: nếu `cnt[t] < c` ⇒ `False`; ngược lại `cnt[t] -= c`.

In [10]:
def divide_counter_sort(nums, k):
    if len(nums) % k:
        return False
    cnt = Counter(nums)
    for x in sorted(cnt):
        c = cnt[x]
        if c == 0:
            continue                       # đã bị nhóm trước dùng hết
        for t in range(x, x + k):
            if cnt[t] < c:
                return False
            cnt[t] -= c
    return True

for nums, k, want in examples:
    print(divide_counter_sort(nums, k), "| mong đợi", want)

True | mong đợi True
True | mong đợi True
False | mong đợi False


In [11]:
class SolutionSort:
    def isPossibleDivide(self, nums: List[int], k: int) -> bool:
        if len(nums) % k:
            return False
        cnt = Counter(nums)
        for x in sorted(cnt):
            c = cnt[x]
            if c:
                for t in range(x, x + k):
                    if cnt[t] < c:
                        return False
                    cnt[t] -= c
        return True

print([SolutionSort().isPossibleDivide(n, k) for n, k, _ in examples])

[True, True, False]


**Độ phức tạp Cách 2:** sort `m` khoá khác nhau `O(m log m)`. Vòng trong chạy `k` bước **chỉ khi** `c > 0`; mỗi lần như vậy rút đi `c·k ≥ k` phần tử, nên có tối đa `n/k` lần ⇒ tổng `O(n)`. Tổng `O(n + m log m)` ≤ `O(n log n)`; bộ nhớ `O(m)`.

### Cách 3 — Min-heap các khoá (phong cách LC 846)

**Ý tưởng:** đưa các khoá vào min-heap. Lặp: `x = heap[0]` là số nhỏ nhất còn lại; rút một nhóm `x … x+k−1` (trừ 1); khi một khoá về 0 thì nó **phải** là đỉnh heap (nếu không ⇒ còn số nhỏ hơn nó chưa dùng hết ⇒ `False`), pop nó.

In [12]:
def divide_heap(nums, k):
    if len(nums) % k:
        return False
    cnt = Counter(nums)
    heap = list(cnt)
    heapq.heapify(heap)
    while heap:
        x = heap[0]
        for t in range(x, x + k):
            if cnt[t] == 0:
                return False
            cnt[t] -= 1
            if cnt[t] == 0:
                if t != heap[0]:           # số nhỏ hơn t vẫn còn -> không thể hoàn thành
                    return False
                heapq.heappop(heap)
    return True

for nums, k, want in examples:
    print(divide_heap(nums, k), "| mong đợi", want)

True | mong đợi True
True | mong đợi True
False | mong đợi False


Vì sao khi `cnt[t]` về 0 mà `t` không phải đỉnh heap thì `False`? Ta đang rút nhóm bắt đầu tại `x` (đỉnh heap lúc bắt đầu), nên `x ≤ t ≤ x + k − 1`. Nếu đỉnh heap `h` khác `t` thì `h < t` và `h` vẫn còn bản sao. Sau này, khi `h` là số nhỏ nhất, nó bắt buộc mở nhóm `h … h+k−1` — nhóm này chứa `t` (vì `h < t ≤ x + k − 1 ≤ h + k − 1`), nhưng `t` đã hết và không bao giờ tăng lại ⇒ không thể chia.

**Độ phức tạp Cách 3:** mỗi phần tử rút đi `O(1)`, mỗi khoá pop một lần `O(log m)` ⇒ `O(n + m log m)`; bộ nhớ `O(m)`. Cách này rút **từng nhóm một**, nên số vòng là `n/k` nhóm × `k` = `O(n)`.

### Cách 4 — Chỉ bắt đầu từ các khoá "đầu chuỗi" (không sort, `O(n)` kỳ vọng)

**Ý tưởng:** nhóm `k` số liên tiếp không bao giờ vắt qua một "khoảng trống" giá trị. Chia các khoá thành các **chuỗi liên tiếp** tối đại; mỗi chuỗi xử lý độc lập. Trong một chuỗi bắt đầu tại `s` (`s − 1` không có), đi `v = s, s+1, …`: tại mỗi `v`, số nhỏ nhất còn lại của chuỗi chính là `v` (các số nhỏ hơn đã về 0) ⇒ áp quy tắc tham lam.

In [13]:
def divide_run_starts(nums, k):
    if len(nums) % k:
        return False
    cnt = Counter(nums)
    for s in list(cnt):
        if s - 1 in cnt:
            continue                      # không phải đầu chuỗi
        v = s
        while v in cnt:                   # đi dọc chuỗi s, s+1, s+2, ...
            c = cnt[v]
            if c:
                for t in range(v, v + k):
                    if cnt[t] < c:        # Counter trả 0 với khoá thiếu, không chèn khoá
                        return False
                    cnt[t] -= c
            v += 1
    return True

for nums, k, want in examples:
    print(divide_run_starts(nums, k), "| mong đợi", want)

True | mong đợi True
True | mong đợi True
False | mong đợi False


Lưu ý: ta không bao giờ xoá khoá khỏi `cnt` (chỉ giảm về 0), nên phép thử `s - 1 in cnt` và `v in cnt` phản ánh **tập giá trị ban đầu**.

**Độ phức tạp Cách 4:** mỗi khoá được "đi qua" đúng một lần (thuộc đúng một chuỗi); vòng trong `k` bước chỉ chạy khi `c > 0`, tối đa `n/k` lần ⇒ `O(n)` kỳ vọng (phép băm), bộ nhớ `O(m)`. Không cần sort!

### Cách 5 — Một lượt với số "nhóm đang mở"

**Ý tưởng:** duyệt khoá tăng dần, giữ `open_cnt` = số nhóm đang mở cần số `x` tiếp theo, và hàng đợi `opened` lưu số nhóm **mới mở** tại mỗi trong `k − 1` khoá gần nhất.
- Nếu còn nhóm mở mà `x ≠ last + 1` ⇒ `False` (nhóm bị đứt).
- Nếu `open_cnt > cnt[x]` ⇒ không đủ `x` cho các nhóm đang mở ⇒ `False`.
- Mở thêm `cnt[x] − open_cnt` nhóm mới tại `x`; khi hàng đợi đủ `k` phần tử, các nhóm mở cách đây `k` bước **hoàn thành** ⇒ đóng lại.

In [14]:
def divide_open_groups(nums, k):
    if len(nums) % k:
        return False
    cnt = Counter(nums)
    opened = deque()                 # số nhóm mở mới tại từng khoá gần đây
    open_cnt, last = 0, None
    for x in sorted(cnt):
        if open_cnt and x != last + 1:
            return False
        if open_cnt > cnt[x]:
            return False
        opened.append(cnt[x] - open_cnt)
        open_cnt = cnt[x]            # mọi bản sao x đều đang thuộc một nhóm mở
        last = x
        if len(opened) == k:
            open_cnt -= opened.popleft()   # nhóm mở cách đây k-1 khoá vừa đủ k phần tử
    return open_cnt == 0

for nums, k, want in examples:
    print(divide_open_groups(nums, k), "| mong đợi", want)

True | mong đợi True
True | mong đợi True
False | mong đợi False


**Độ phức tạp Cách 5:** `O(n + m log m)` thời gian (đếm + sort khoá), mỗi khoá xử lý `O(1)`; bộ nhớ `O(m + k)`. Độc lập với `k` — tốt khi `k` lớn.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2** `[3,2,1,2,3,4,3,4,5,9,10,11]`, `k = 3`. Đếm ban đầu `{1:1, 2:2, 3:3, 4:2, 5:1, 9:1, 10:1, 11:1}`.

| x | cnt[x] = c | nhóm cần | kiểm tra | đếm sau khi trừ |
|---|---|---|---|---|
| 1 | 1 | 1,2,3 | đủ | {2:1, 3:2, 4:2, 5:1, …} |
| 2 | 1 | 2,3,4 | đủ | {3:1, 4:1, 5:1, …} |
| 3 | 1 | 3,4,5 | đủ | {9:1, 10:1, 11:1} |
| 4, 5 | 0 | — | bỏ qua | |
| 9 | 1 | 9,10,11 | đủ | rỗng ⇒ **True** |

In [15]:
def trace_counter(nums, k):
    if len(nums) % k:
        print("len % k != 0 -> False"); return False
    cnt = Counter(nums)
    print("đếm ban đầu:", dict(sorted(cnt.items())))
    for x in sorted(cnt):
        c = cnt[x]
        if c == 0:
            print(f"x={x}: cnt=0, bỏ qua")
            continue
        need = list(range(x, x + k))
        have = [cnt[t] for t in need]
        if any(h < c for h in have):
            print(f"x={x}: cần {c} nhóm {need}, đang có {have} -> THIẾU -> False")
            return False
        for t in need:
            cnt[t] -= c
        print(f"x={x}: mở {c} nhóm {need} -> còn {dict((t, v) for t, v in sorted(cnt.items()) if v)}")
    return True

print("đáp án =", trace_counter(*examples[1][:2]))
print()
print("đáp án =", trace_counter([1, 2, 3, 6, 2, 3, 4, 7, 8], 3))

đếm ban đầu: {1: 1, 2: 2, 3: 3, 4: 2, 5: 1, 9: 1, 10: 1, 11: 1}
x=1: mở 1 nhóm [1, 2, 3] -> còn {2: 1, 3: 2, 4: 2, 5: 1, 9: 1, 10: 1, 11: 1}
x=2: mở 1 nhóm [2, 3, 4] -> còn {3: 1, 4: 1, 5: 1, 9: 1, 10: 1, 11: 1}
x=3: mở 1 nhóm [3, 4, 5] -> còn {9: 1, 10: 1, 11: 1}
x=4: cnt=0, bỏ qua
x=5: cnt=0, bỏ qua
x=9: mở 1 nhóm [9, 10, 11] -> còn {}
x=10: cnt=0, bỏ qua
x=11: cnt=0, bỏ qua
đáp án = True

đếm ban đầu: {1: 1, 2: 2, 3: 2, 4: 1, 6: 1, 7: 1, 8: 1}
x=1: mở 1 nhóm [1, 2, 3] -> còn {2: 1, 3: 1, 4: 1, 6: 1, 7: 1, 8: 1}
x=2: mở 1 nhóm [2, 3, 4] -> còn {6: 1, 7: 1, 8: 1}
x=3: cnt=0, bỏ qua
x=4: cnt=0, bỏ qua
x=6: mở 1 nhóm [6, 7, 8] -> còn {}
x=7: cnt=0, bỏ qua
x=8: cnt=0, bỏ qua
đáp án = True


In [16]:
def trace_open_groups(nums, k):
    cnt = Counter(nums)
    opened, open_cnt, last = deque(), 0, None
    for x in sorted(cnt):
        if open_cnt and x != last + 1:
            print(f"x={x}: còn {open_cnt} nhóm mở nhưng x != {last}+1 -> False"); return False
        if open_cnt > cnt[x]:
            print(f"x={x}: cần {open_cnt} số {x} nhưng chỉ có {cnt[x]} -> False"); return False
        opened.append(cnt[x] - open_cnt)
        open_cnt, last = cnt[x], x
        msg = f"x={x}: cnt={cnt[x]}, mở mới {opened[-1]}, đang mở {open_cnt}, hàng đợi {list(opened)}"
        if len(opened) == k:
            done = opened.popleft()
            open_cnt -= done
            msg += f" -> hoàn thành {done} nhóm, còn mở {open_cnt}"
        print(msg)
    print("kết thúc: open_cnt =", open_cnt)
    return open_cnt == 0

print("đáp án =", trace_open_groups(*examples[0][:2]))

x=1: cnt=1, mở mới 1, đang mở 1, hàng đợi [1]
x=2: cnt=1, mở mới 0, đang mở 1, hàng đợi [1, 0]
x=3: cnt=2, mở mới 1, đang mở 2, hàng đợi [1, 0, 1]
x=4: cnt=2, mở mới 0, đang mở 2, hàng đợi [1, 0, 1, 0] -> hoàn thành 1 nhóm, còn mở 1
x=5: cnt=1, mở mới 0, đang mở 1, hàng đợi [0, 1, 0, 0] -> hoàn thành 0 nhóm, còn mở 1
x=6: cnt=1, mở mới 0, đang mở 1, hàng đợi [1, 0, 0, 0] -> hoàn thành 1 nhóm, còn mở 0
kết thúc: open_cnt = 0
đáp án = True


## 9. Code nộp LeetCode

Chọn Cách 2 (Counter + khoá sắp xếp): ngắn, dễ chứng minh, `O(n log n)`.

In [17]:
class Solution:
    def isPossibleDivide(self, nums: List[int], k: int) -> bool:
        if len(nums) % k != 0:
            return False
        cnt = Counter(nums)
        for x in sorted(cnt):
            c = cnt[x]
            if c == 0:
                continue
            # c bản sao của x đều phải mở đầu nhóm x, x+1, ..., x+k-1
            for t in range(x, x + k):
                if cnt[t] < c:
                    return False
                cnt[t] -= c
        return True

for nums, k, want in examples:
    got = Solution().isPossibleDivide(nums, k)
    print(got, "OK" if got == want else "SAI")

True OK
True OK
False OK


## 10. Vì sao thuật toán đúng

**Bổ đề tham lam:** gọi `x` là giá trị nhỏ nhất trong multiset `M`, xuất hiện `c` lần. Nếu `M` chia được thành các nhóm `k` số liên tiếp, thì trong **mọi** cách chia, mỗi bản sao của `x` nằm trong một nhóm mà `x` là phần tử nhỏ nhất (không có giá trị nào nhỏ hơn) ⇒ nhóm đó đúng là `[x, …, x+k−1]`. Hai bản sao của `x` không thể chung nhóm (nhóm gồm các số **khác nhau**). Vậy mọi cách chia đều chứa đúng `c` nhóm `[x, …, x+k−1]`.

**Hệ quả:** `M` chia được ⇔ `cnt[x+i] ≥ c` với mọi `i < k` **và** `M' = M − c·{x, …, x+k−1}` chia được.
- (⇒) theo bổ đề.
- (⇐) ghép `c` nhóm vừa lấy với cách chia của `M'`.

Thuật toán áp dụng đúng hệ quả này lặp lại, với `x` lần lượt là khoá nhỏ nhất còn `cnt > 0` (duyệt `sorted(cnt)` và bỏ qua khoá đã về 0). Kết thúc khi mọi khoá về 0 ⇒ `True`; dừng sớm khi thiếu ⇒ `False` là đúng.

**Cách 4** đúng vì nhóm không thể chứa hai giá trị thuộc hai chuỗi khác nhau (giữa chúng có giá trị bị thiếu), nên các chuỗi độc lập; trong mỗi chuỗi, khi xét `v` thì mọi giá trị nhỏ hơn trong chuỗi đã về 0 ⇒ `v` là nhỏ nhất ⇒ áp bổ đề.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `m` = số giá trị khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force (list.remove) | `O(n²)` | `O(n)` |
| 2. Counter + sort khoá | `O(n + m log m)` | `O(m)` |
| 3. Counter + heap | `O(n + m log m)` | `O(m)` |
| 4. Đầu chuỗi (không sort) | `O(n)` kỳ vọng | `O(m)` |
| 5. Nhóm đang mở + deque | `O(n + m log m)` | `O(m + k)` |

Phần `O(n)` ở Cách 2, 4: vòng `k` bước chỉ chạy khi có `c > 0`, mỗi lần rút ít nhất `k` phần tử ⇒ tối đa `n/k` lần.

## 12. Bộ test

Ví dụ đề; edge case (`k = 1`, `k = n`, trùng lặp nhiều, giá trị `10^9`, `n % k ≠ 0`, khoảng trống giá trị); đối chiếu ngẫu nhiên với một **vét cạn độc lập** (thử mọi cách chọn nhóm chứa phần tử đầu tiên — không dùng lập luận tham lam) trên mảng nhỏ; và test hiệu năng `n = 10^5`.

In [18]:
def divide_exhaustive(nums, k):
    """Vét cạn độc lập: phần tử nums[0] của phần còn lại phải nằm trong một nhóm nào đó;
    thử mọi cách chọn k-1 phần tử còn lại để cùng nhóm."""
    if len(nums) % k:
        return False
    def go(rest):
        if not rest:
            return True
        first, others = rest[0], rest[1:]
        tried = set()
        for idx in combinations(range(len(others)), k - 1):
            group = sorted([first] + [others[i] for i in idx])
            key = tuple(group)
            if key in tried:
                continue
            tried.add(key)
            if all(group[i + 1] - group[i] == 1 for i in range(k - 1)):
                chosen = set(idx)
                if go([others[i] for i in range(len(others)) if i not in chosen]):
                    return True
        return False
    return go(list(nums))

ALL = {
    "brute": divide_brute, "counter_sort": divide_counter_sort, "heap": divide_heap,
    "run_starts": divide_run_starts, "open_groups": divide_open_groups,
    "SolSort": lambda a, k: SolutionSort().isPossibleDivide(a, k),
    "Solution": lambda a, k: Solution().isPossibleDivide(a, k),
}
fixed = [
    ([1, 2, 3, 3, 4, 4, 5, 6], 4, True),
    ([3, 2, 1, 2, 3, 4, 3, 4, 5, 9, 10, 11], 3, True),
    ([1, 2, 3, 4], 3, False),
    ([5], 1, True), ([5, 5, 5], 1, True),          # k = 1 luôn True
    ([4, 3, 2, 1], 4, True), ([1, 2, 4, 5], 4, False),   # k = n
    ([1, 1, 2, 2, 3, 3], 3, True), ([1, 1, 2, 3, 3, 4], 3, False),
    ([1, 3, 5, 7], 2, False), ([1, 2, 4, 5], 2, True),
    ([10**9 - 1, 10**9], 2, True), ([1, 2, 3, 6, 2, 3, 4, 7, 8], 3, True),
    ([1, 2, 3, 2, 3, 4, 3, 4, 5, 5, 6, 7], 3, True),
]
for nums, k, want in fixed:
    snap = nums[:]
    assert divide_exhaustive(nums, k) == want, ("exhaustive", nums, k)
    for name, f in ALL.items():
        got = f(nums, k)
        assert got == want, (name, nums, k, got, want)
    assert nums == snap
print("Test cố định: qua", len(fixed), "case (kể cả vét cạn độc lập), input không bị sửa")

Test cố định: qua 14 case (kể cả vét cạn độc lập), input không bị sửa


In [19]:
rng = random.Random(1296)
true_cnt = 0
for _ in range(1200):
    k = rng.randint(1, 4)
    if rng.random() < 0.5:                         # sinh mảng chắc chắn chia được, rồi có thể làm hỏng
        groups = rng.randint(1, 3)
        nums = []
        for _ in range(groups):
            s = rng.randint(1, 6)
            nums += list(range(s, s + k))
        if rng.random() < 0.4:
            nums[rng.randrange(len(nums))] += rng.choice([-1, 1, 2])
    else:
        nums = [rng.randint(1, 7) for _ in range(rng.randint(1, 9))]
    rng.shuffle(nums)
    want = divide_exhaustive(nums, k)
    true_cnt += want
    for name, f in ALL.items():
        assert f(nums, k) == want, (name, nums, k)
print(f"1200 test ngẫu nhiên: mọi cách khớp vét cạn độc lập ({true_cnt} test có đáp án True)")

1200 test ngẫu nhiên: mọi cách khớp vét cạn độc lập (588 test có đáp án True)


In [20]:
n = 10**5
k = 5
base = [rng.randint(1, 10**9 - k) for _ in range(n // k)]
big_true = [s + i for s in base for i in range(k)]
rng.shuffle(big_true)
big_false = big_true[:]
big_false[0] += 1                                  # rất có thể làm hỏng
for name in ["counter_sort", "heap", "run_starts", "open_groups", "Solution"]:
    t = time.perf_counter()
    a, b = ALL[name](big_true, k), ALL[name](big_false, k)
    print(f"{name:>12}: {a}, {b}, {time.perf_counter() - t:.3f}s")

counter_sort: True, False, 0.140s
        heap: True, False, 0.158s


  run_starts: True, False, 0.059s
 open_groups: True, False, 0.092s
    Solution: True, False, 0.104s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Duyệt khoá của `Counter` theo thứ tự chèn (không sort)

In [21]:
def wrong_unsorted(nums, k):
    if len(nums) % k:
        return False
    cnt = Counter(nums)
    for x in cnt:                                  # SAI: không phải thứ tự tăng
        c = cnt[x]
        if c:
            for t in range(x, x + k):
                if cnt[t] < c:
                    return False
                cnt[t] -= c
    return True

nums, k, want = examples[1]
print("Code sai:", wrong_unsorted(nums, k), "| đúng là:", want)

Code sai: False | đúng là: True


Khoá đầu tiên theo thứ tự chèn là `3`, không phải số nhỏ nhất `1` ⇒ cố mở 3 nhóm `[3,4,5]` và thất bại. **Code đúng:** `for x in sorted(cnt)` (hoặc dùng heap / đầu chuỗi).

### Lỗi 2 — Mỗi khoá chỉ rút **một** nhóm

In [22]:
def wrong_one_group(nums, k):
    if len(nums) % k:
        return False
    cnt = Counter(nums)
    for x in sorted(cnt):
        if cnt[x] > 0:
            for t in range(x, x + k):
                if cnt[t] < 1:
                    return False
                cnt[t] -= 1                        # SAI: còn cnt[x] - 1 bản sao của x chưa xử lý
    return all(v == 0 for v in cnt.values())

print("Code sai:", wrong_one_group([1, 1, 2, 2, 3, 3], 3), "| đúng là:", Solution().isPossibleDivide([1, 1, 2, 2, 3, 3], 3))

Code sai: False | đúng là: True


Sau khi rút `[1,2,3]` một lần, bản sao thứ hai của `1` bị bỏ qua (vòng `for` đã đi tiếp sang `2`). **Code đúng:** rút `c = cnt[x]` nhóm cùng lúc (Cách 2), hoặc lặp `while cnt[x] > 0`.

### Lỗi 3 — Dùng `dict` thường ⇒ `KeyError` khi số cần không tồn tại

In [23]:
def wrong_plain_dict(nums, k):
    if len(nums) % k:
        return False
    cnt = {}
    for x in nums:
        cnt[x] = cnt.get(x, 0) + 1
    for x in sorted(cnt):
        c = cnt[x]
        if c:
            for t in range(x, x + k):
                if cnt[t] < c:                     # SAI: KeyError nếu t không có
                    return False
                cnt[t] -= c
    return True

try:
    print(wrong_plain_dict([1, 2, 4, 5], 2), wrong_plain_dict([1, 3], 2))
except KeyError as e:
    print("Code sai gặp lỗi: KeyError", e)
print("Code đúng:", Solution().isPossibleDivide([1, 3], 2))

Code sai gặp lỗi: KeyError 2
Code đúng: False


**Code đúng:** dùng `Counter` (trả 0 với khoá thiếu) hoặc `cnt.get(t, 0)`.

### Lỗi 4 — Chỉ kiểm tra điều kiện cần (`n % k == 0` và đủ số khác nhau)

In [24]:
def wrong_necessary_only(nums, k):
    return len(nums) % k == 0 and len(set(nums)) >= k     # SAI: cần nhưng không đủ

for nums, k in [([1, 3, 5, 7], 2), ([1, 2, 2, 4], 2)]:
    print(f"Code sai {nums}, k={k}:", wrong_necessary_only(nums, k),
          "| đúng là:", Solution().isPossibleDivide(nums, k))

Code sai [1, 3, 5, 7], k=2: True | đúng là: False
Code sai [1, 2, 2, 4], k=2: True | đúng là: False


**Code đúng:** phải thực sự mô phỏng việc chia (tham lam từ số nhỏ nhất). `n % k == 0` chỉ là bước lọc nhanh.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chia thành các nhóm **liên tiếp**" ⇒ nghĩ ngay: **phần tử nhỏ nhất bắt buộc mở nhóm** ⇒ tham lam + `Counter`.
- Luôn lọc nhanh `len(nums) % k`.
- Rút **cả loạt** `cnt[x]` nhóm một lần để code gọn và nhanh.
- Không muốn sort ⇒ bắt đầu từ các **đầu chuỗi** (`x − 1` không có) — cùng mẹo với LC 128 *Longest Consecutive Sequence*.
- `k = 1` ⇒ luôn `True`; `k = n` ⇒ mảng phải là một dãy liên tiếp không lặp.

## 15. Biến thể của bài

1. **LC 846 Hand of Straights:** y hệt (tên tham số `hand`, `groupSize`).
2. **LC 659 Split Array into Consecutive Subsequences:** độ dài nhóm **≥ 3** (không cố định), mảng đã sắp xếp — tham lam với hai bảng `cnt` và `end` (Bài tập 5).
3. **Trả về các nhóm** thay vì `True/False` (Bài tập 3).
4. **LC 128 Longest Consecutive Sequence:** mẹo "đầu chuỗi" `O(n)`.
5. **LC 2007 Find Original Array From Doubled Array:** cũng là "phần tử nhỏ nhất bắt buộc ghép với…" + `Counter`.
6. **Nhóm có bước nhảy `d`** (`x, x+d, …`): thay `x + i` bằng `x + i·d`.

In [25]:
def longest_consecutive(nums):            # LC 128 — mẹo đầu chuỗi
    s = set(nums)
    best = 0
    for x in s:
        if x - 1 not in s:
            y = x
            while y in s:
                y += 1
            best = max(best, y - x)
    return best

print(longest_consecutive([100, 4, 200, 1, 3, 2]), longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]))

4 9


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | sort, lấy nhỏ nhất, `list.remove` | `O(n²)` | `O(n)` | hiểu bài, `n` nhỏ |
| 2. Counter + sort khoá | rút `cnt[x]` nhóm từ khoá nhỏ nhất | `O(n + m log m)` | `O(m)` | **nộp bài** |
| 3. Counter + heap | đỉnh heap = nhỏ nhất, rút từng nhóm | `O(n + m log m)` | `O(m)` | phong cách LC 846 phổ biến |
| 4. Đầu chuỗi | xử lý từng chuỗi giá trị liên tiếp | `O(n)` kỳ vọng | `O(m)` | không muốn sort |
| 5. Nhóm đang mở | một lượt, deque độ dài `k` | `O(n + m log m)` | `O(m + k)` | `k` lớn, muốn mỗi khoá `O(1)` |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `isPossibleDivide([1,2,3,4], 2)`, `isPossibleDivide([1,1,2,2,3,3], 3)`, `isPossibleDivide([1,2,4], 3)`, `isPossibleDivide([5], 1)`?

**Bài 2 (tính tay):** Trace Cách 2 trên `[1,2,3,6,2,3,4,7,8]`, `k = 3`: ghi đếm sau mỗi bước.

**Bài 3 (điền TODO):** Hoàn thiện `divide_groups_todo(nums, k)` trả về **danh sách các nhóm** (mỗi nhóm là list tăng dần), hoặc `[]` nếu không chia được.

**Bài 4 (tự viết lại):** Tự viết lại Cách 3 (heap) — không nhìn mục 7.

**Bài 5 (mini challenge):** LC 659 — cho `nums` **đã sắp xếp tăng**, có thể chia thành các dãy con liên tiếp, mỗi dãy **dài ít nhất 3** hay không? Viết `split_consecutive(nums)` `O(n)`.

In [26]:
def divide_groups_todo(nums, k):
    if len(nums) % k:
        return []
    cnt = Counter(nums)
    groups = []
    for x in sorted(cnt):
        c = cnt[x]
        # TODO: nếu c > 0, kiểm tra đủ, trừ c, và thêm c bản sao của list(range(x, x + k)) vào groups
    return groups

def split_consecutive_todo(nums):
    cnt = Counter(nums)
    end = Counter()        # end[v] = số dãy đang kết thúc tại v
    # TODO: với mỗi x: ưu tiên nối vào dãy kết thúc tại x-1; nếu không, mở dãy mới x, x+1, x+2
    pass

# Lưu ý: cell này chỉ định nghĩa hàm, không gọi.

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Kiểm tra `n % k` trước. (2) Áp dụng "số nhỏ nhất mở nhóm". (3) `[1,2,4]`, `k=3`: mở `[1,2,3]` thiếu `3`.

**Bài 2.** (1) Đếm `{1:1, 2:2, 3:2, 4:1, 6:1, 7:1, 8:1}`. (2) `x=1` ⇒ `[1,2,3]`; `x=2` ⇒ `[2,3,4]`; `x=6` ⇒ `[6,7,8]`. (3) Kết quả `True`.

**Bài 3.** (1) Giống Cách 2. (2) `groups.extend([list(range(x, x + k)) for _ in range(c)])`. (3) Kiểm tra: ghép mọi nhóm lại phải bằng `nums` sau khi sort.

**Bài 4.** (1) `heap = list(cnt)`, `heapify`. (2) `x = heap[0]`, rút `x..x+k−1` mỗi số 1 lần. (3) Khi một khoá về 0 mà không phải đỉnh heap ⇒ `False`.

**Bài 5.** (1) Duyệt `x` theo thứ tự; bỏ qua nếu `cnt[x] == 0`. (2) Nếu `end[x−1] > 0`: nối `x` vào (`end[x−1] −= 1`, `end[x] += 1`). (3) Ngược lại cần `cnt[x+1], cnt[x+2] > 0` để mở dãy mới `x, x+1, x+2` (`end[x+2] += 1`); không được ⇒ `False`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().isPossibleDivide([1, 2, 3, 4], 2) is True
assert Solution().isPossibleDivide([1, 1, 2, 2, 3, 3], 3) is True
assert Solution().isPossibleDivide([1, 2, 4], 3) is False
assert Solution().isPossibleDivide([5], 1) is True

# Bài 2
assert Solution().isPossibleDivide([1, 2, 3, 6, 2, 3, 4, 7, 8], 3) is True

# Bài 3
def divide_groups_sol(nums, k):
    if len(nums) % k:
        return []
    cnt = Counter(nums)
    groups = []
    for x in sorted(cnt):
        c = cnt[x]
        if c:
            for t in range(x, x + k):
                if cnt[t] < c:
                    return []
                cnt[t] -= c
            groups.extend([list(range(x, x + k)) for _ in range(c)])
    return groups

g = divide_groups_sol([1, 2, 3, 3, 4, 4, 5, 6], 4)
assert g == [[1, 2, 3, 4], [3, 4, 5, 6]]

# Bài 4
def divide_heap_sol(nums, k):
    if len(nums) % k:
        return False
    cnt = Counter(nums)
    heap = list(cnt)
    heapq.heapify(heap)
    while heap:
        x = heap[0]
        for t in range(x, x + k):
            if cnt[t] == 0:
                return False
            cnt[t] -= 1
            if cnt[t] == 0:
                if heap[0] != t:
                    return False
                heapq.heappop(heap)
    return True

# Bài 5
def split_consecutive_sol(nums):
    cnt, end = Counter(nums), Counter()
    for x in nums:
        if cnt[x] == 0:
            continue
        cnt[x] -= 1
        if end[x - 1] > 0:
            end[x - 1] -= 1
            end[x] += 1
        elif cnt[x + 1] > 0 and cnt[x + 2] > 0:
            cnt[x + 1] -= 1
            cnt[x + 2] -= 1
            end[x + 2] += 1
        else:
            return False
    return True

def split_consecutive_brute(nums):
    """Vét cạn: mỗi phần tử (theo thứ tự) nối vào một dãy đang kết thúc ở x-1, hoặc mở dãy mới."""
    def go(i, seqs):
        if i == len(nums):
            return all(length >= 3 for _, length in seqs)
        x = nums[i]
        for j, (last, length) in enumerate(seqs):
            if last == x - 1:
                if go(i + 1, seqs[:j] + [(x, length + 1)] + seqs[j + 1:]):
                    return True
        return go(i + 1, seqs + [(x, 1)])
    return go(0, [])

assert split_consecutive_sol([1, 2, 3, 3, 4, 5]) is True          # ví dụ LC 659
assert split_consecutive_sol([1, 2, 3, 3, 4, 4, 5, 5]) is True
assert split_consecutive_sol([1, 2, 3, 4, 4, 5]) is False

rng = random.Random(659)
for _ in range(600):
    nums = sorted(rng.randint(1, 6) for _ in range(rng.randint(1, 8)))
    k = rng.randint(1, 4)
    want = divide_exhaustive(nums, k)
    gs = divide_groups_sol(nums, k)
    assert (gs != []) == want
    if want:
        assert sorted(x for grp in gs for x in grp) == nums
        assert all(len(grp) == k and grp == list(range(grp[0], grp[0] + k)) for grp in gs)
    assert divide_heap_sol(nums, k) == want
    assert split_consecutive_sol(nums) == split_consecutive_brute(nums)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "phần tử nhỏ nhất mở nhóm":**

```python
if len(nums) % k: return False
cnt = Counter(nums)
for x in sorted(cnt):
    c = cnt[x]
    if c == 0: continue
    for t in range(x, x + k):
        if cnt[t] < c: return False
        cnt[t] -= c
return True
```

**Template đầu chuỗi (không sort):**

```python
for s in list(cnt):
    if s - 1 in cnt: continue
    v = s
    while v in cnt:
        ...   # v là nhỏ nhất còn lại trong chuỗi
        v += 1
```

**Checklist trước khi submit:**
- [ ] Lọc `len(nums) % k != 0`.
- [ ] Duyệt khoá **tăng dần** (hoặc heap / đầu chuỗi).
- [ ] Rút `cnt[x]` nhóm cùng lúc; bỏ qua khoá đã về 0.
- [ ] Dùng `Counter` (hoặc `.get(t, 0)`) để không `KeyError`.
- [ ] Không xoá khoá khỏi `Counter` nếu dùng phép thử `in` để nhận diện đầu chuỗi.
- [ ] Đúng chữ ký: `isPossibleDivide(self, nums: List[int], k: int) -> bool`.